# 📝 EDA·시각화 과제 LV1 정답 — 결합·집계·기본 그래프 (강사용)

각 문제의 **모범답안 + 해설(접근법·흔한 실수·대안)** 입니다. 학생이 스스로 풀어 본 뒤 비교하도록 안내하세요.

- 경로는 정답 노트북 기준 `../../day07_EDA_시각화/data/` 입니다.
- 그래프 문제는 자가채점이 없습니다 — 문제 설명 아래 **완성 그래프(정답)** 와 같은 모양으로 그리면 됩니다.

아래 셀을 먼저 실행해 시각화 라이브러리와 한글 폰트를 준비하세요.

In [ ]:
# [제공 코드] 시각화 라이브러리와 한글 폰트를 준비합니다.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
문제를 풀기 전에 **어떤 데이터인지 먼저 파악**합니다. `head()` 로 앞부분을, `info()` 로 열·자료형·결측을, `describe()` 로 수치 요약을 봅니다. (아래 셀은 실행만 하면 됩니다.)

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다 — 앞부분·구조·수치 요약
df = pd.read_csv('../../day07_EDA_시각화/data/titanic.csv')
print("행·열 크기:", df.shape)
print("\n[앞 5행] head()"); display(df.head())
print("\n[열·자료형·결측] info()"); df.info()
print("\n[수치 요약] describe()"); display(df.describe())

## 1. 그룹별 평균 (groupby)
**배경**: 객실 등급(`pclass`)이 좋을수록 요금(`fare`)이 비쌌을까요? 등급별로 묶어 평균을 내면 한눈에 보입니다.

**요구사항**:
- `data/titanic.csv` 파일을 읽어 변수 `df` 에 담으세요.
- `pclass` 로 묶어 `fare` 의 **평균**을 구해 `avg_fare_by_class` 에 담으세요. (결과는 Series, 인덱스는 `pclass`)

**예시**
```
avg_fare_by_class.loc[1]  →  84.15...   (1등석 평균 요금)
len(avg_fare_by_class)    →  3          (등급 1·2·3)
```
<details><summary>힌트</summary>

```text
접근방법:
- 등급별로 묶은 뒤, 그 그룹의 fare 평균을 낸다.

세부구현:
1. 파일을 df 로 불러온다
2. groupby 로 등급(pclass)별로 묶는다
3. 묶은 결과에서 fare 열의 평균(mean)을 내 avg_fare_by_class 에 담는다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day07_EDA_시각화/data/titanic.csv')
avg_fare_by_class = df.groupby('pclass')['fare'].mean()
print(avg_fare_by_class)

In [ ]:
# [자가채점]
assert round(avg_fare_by_class.loc[1], 2) == 84.15
assert round(avg_fare_by_class.loc[3], 2) == 13.68
assert len(avg_fare_by_class) == 3
print("✅ 문제1 통과!")

### 해설 — 문제 1
- **접근법**: `groupby('pclass')` 는 같은 등급끼리 묶고, 이어지는 `['fare'].mean()` 은 그 묶음마다 요금 평균을 냅니다. 결과는 인덱스가 등급인 Series 예요.
- **흔한 실수**: 묶기만 하고(`df.groupby('pclass')`) 집계 함수를 안 붙이면 그룹 객체만 남습니다. 반드시 `.mean()` 같은 집계를 이어 붙이세요.
- **대안**: `df.groupby('pclass')['fare'].agg('mean')` 도 같은 결과입니다.

## 2. 그룹별 합계로 개수 세기 (groupby + sum)
**배경**: `survived` 는 생존이면 1, 사망이면 0 입니다. 성별로 묶어 `survived` 를 **합**하면 곧 성별 생존자 수가 됩니다.

**요구사항**:
- `data/titanic.csv` 파일을 읽어 변수 `df` 에 담으세요.
- `sex` 로 묶어 `survived` 의 **합계**를 구해 `survivors_by_sex` 에 담으세요. (결과는 Series, 인덱스는 `sex`)

**예시**
```
survivors_by_sex.loc['female']  →  233
survivors_by_sex.loc['male']    →  109
```
<details><summary>힌트</summary>

```text
접근방법:
- 0/1 로 된 열은 합계가 곧 1의 개수(=생존자 수)다.

세부구현:
1. 파일을 df 로 불러온다
2. groupby 로 성별(sex)로 묶는다
3. survived 열의 합(sum)을 내 survivors_by_sex 에 담는다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day07_EDA_시각화/data/titanic.csv')
survivors_by_sex = df.groupby('sex')['survived'].sum()
print(survivors_by_sex)

In [ ]:
# [자가채점]
assert survivors_by_sex.loc['female'] == 233
assert survivors_by_sex.loc['male'] == 109
print("✅ 문제2 통과!")

### 해설 — 문제 2
- **접근법**: `survived` 가 0/1 이라, 합계(`sum`)는 1(생존)의 개수와 같습니다. 성별로 묶어 합하면 성별 생존자 수예요.
- **흔한 실수**: 여기서 `mean` 을 쓰면 생존자 **수**가 아니라 생존 **비율**이 나옵니다. "몇 명"을 원하면 `sum` 입니다.
- **대안**: 각 성별의 전체 인원은 `df.groupby('sex')['survived'].count()`, 생존 비율은 `mean()` 으로 볼 수 있어요.

## 3. 한 번에 여러 통계 (agg)
**배경**: 등급별 나이(`age`)의 평균만이 아니라 최댓값·최솟값까지 한 번에 보고 싶을 때 `agg` 에 함수 목록을 넘깁니다.

**요구사항**:
- `data/titanic.csv` 파일을 읽어 변수 `df` 에 담으세요.
- `pclass` 로 묶어 `age` 의 **평균·최댓값·최솟값**을 한 번에 구해 `age_stats` 에 담으세요.
- 집계는 `agg(['mean', 'max', 'min'])` 을 사용하세요. (결과는 DataFrame, 열은 `mean`·`max`·`min`)

**예시**
```
list(age_stats.columns)      →  ['mean', 'max', 'min']
age_stats.loc[1, 'max']      →  80.0
```
<details><summary>힌트</summary>

```text
접근방법:
- 묶은 뒤 agg 에 함수 이름 목록을 주면 열마다 통계가 나온다.

세부구현:
1. 파일을 df 로 불러온다
2. groupby 로 등급(pclass)별 나이(age)를 묶는다
3. agg 에 mean·max·min 세 함수 이름을 목록으로 주어 age_stats 에 담는다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day07_EDA_시각화/data/titanic.csv')
age_stats = df.groupby('pclass')['age'].agg(['mean', 'max', 'min'])
print(age_stats)

In [ ]:
# [자가채점]
assert list(age_stats.columns) == ['mean', 'max', 'min']
assert round(age_stats.loc[1, 'mean'], 2) == 38.23
assert age_stats.loc[1, 'max'] == 80.0
assert age_stats.loc[3, 'min'] == 0.42
print("✅ 문제3 통과!")

### 해설 — 문제 3
- **접근법**: `agg` 에 함수 이름 목록을 주면 각 함수가 하나의 열이 됩니다. 한 번의 집계로 여러 통계를 얻어요.
- **흔한 실수**: 함수 이름은 문자열 목록(`['mean','max','min']`)으로 넘깁니다. `agg('mean','max')` 처럼 콤마로 나열하면 안 돼요.
- **대안**: 열마다 다른 통계를 원하면 딕셔너리로 `agg({'age':'mean', 'fare':'max'})` 처럼 줄 수도 있습니다.

## 4. 두 기준으로 요약 (pivot_table)
**배경**: "등급 × 성별" 처럼 **행 기준·열 기준 두 축**으로 값을 요약할 때 `pivot_table` 이 편합니다. 등급·성별 생존율 표를 만들어 봅시다.

**요구사항**:
- `data/titanic.csv` 파일을 읽어 변수 `df` 에 담으세요.
- `index='pclass'`, `columns='sex'`, `values='survived'` 로 **평균**(생존율)을 요약해 `pv` 에 담으세요.
- `pivot_table` 의 기본 집계는 평균이므로 `aggfunc` 는 생략해도 됩니다.

**예시**
```
pv.shape            →  (3, 2)     (등급 3 × 성별 2)
pv.loc[1, 'female'] →  0.968...   (1등석 여성 생존율)
```
<details><summary>힌트</summary>

```text
접근방법:
- index·columns·values 세 축을 지정하면 교차 요약표가 나온다.

세부구현:
1. 파일을 df 로 불러온다
2. pivot_table 로 행은 등급(pclass), 열은 성별(sex), 값은 생존여부(survived)가 되게 요약한다
3. 결과를 pv 에 담는다 (기본 집계가 평균이라 aggfunc 생략 가능)
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day07_EDA_시각화/data/titanic.csv')
pv = df.pivot_table(index='pclass', columns='sex', values='survived')
print(pv)

In [ ]:
# [자가채점]
assert pv.shape == (3, 2)
assert round(pv.loc[1, 'female'], 3) == 0.968
assert round(pv.loc[3, 'male'], 3) == 0.135
print("✅ 문제4 통과!")

### 해설 — 문제 4
- **접근법**: `pivot_table` 은 `index`(행 기준), `columns`(열 기준), `values`(요약할 값) 세 축으로 교차표를 만듭니다. 기본 집계는 평균이라 생존율이 바로 나와요.
- **흔한 실수**: `values` 를 빼면 나머지 모든 숫자 열을 요약하려 해 표가 넓어집니다. 요약할 열을 꼭 지정하세요.
- **대안**: 합계로 보고 싶으면 `aggfunc='sum'` 을 줍니다. 다음 문제의 `crosstab` 은 개수 요약에 특화된 도구예요.

## 5. 빈도 교차표 (crosstab)
**배경**: 두 범주가 **몇 번씩 함께 나타나는지**(빈도)만 세고 싶을 때는 `pd.crosstab` 이 가장 간단합니다. 등급별 생존/사망 인원표를 만들어 봅시다.

**요구사항**:
- `data/titanic.csv` 파일을 읽어 변수 `df` 에 담으세요.
- `pd.crosstab` 으로 행은 `pclass`, 열은 `survived` 인 **빈도표**를 만들어 `ct` 에 담으세요.

**예시**
```
ct.shape        →  (3, 2)     (등급 3 × 생존여부 2)
ct.loc[1, 1]    →  136        (1등석 생존자 수)
ct.loc[3, 0]    →  372        (3등석 사망자 수)
```
<details><summary>힌트</summary>

```text
접근방법:
- crosstab 은 두 열을 받아 함께 나타난 횟수를 센다.

세부구현:
1. 파일을 df 로 불러온다
2. crosstab 에 첫 인자로 pclass 열, 둘째 인자로 survived 열을 주어 빈도표를 만든다
3. 결과를 ct 에 담는다 (첫 인자가 행, 둘째가 열)
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day07_EDA_시각화/data/titanic.csv')
ct = pd.crosstab(df['pclass'], df['survived'])
print(ct)

In [ ]:
# [자가채점]
assert ct.shape == (3, 2)
assert ct.loc[1, 1] == 136
assert ct.loc[3, 0] == 372
print("✅ 문제5 통과!")

### 해설 — 문제 5
- **접근법**: `pd.crosstab(행, 열)` 은 두 범주가 함께 나타난 **횟수**를 세어 표로 만듭니다. 별도의 집계 함수가 필요 없어요.
- **흔한 실수**: `pivot_table` 과 헷갈리기 쉬운데, `crosstab` 은 `df.` 이 아니라 `pd.` 로 호출하고 열(Series)을 직접 넘깁니다.
- **대안**: `df.pivot_table(index='pclass', columns='survived', values='survived', aggfunc='count')` 로도 같은 빈도표를 만들 수 있지만 `crosstab` 이 훨씬 짧습니다.

## 6. 표 붙이기 — 열 방향 (merge)
**배경**: 탑승 항구 코드(`embarked`: S·C·Q)만으로는 무슨 항구인지 알기 어렵습니다. 항구 정보표(`port_info.csv`)를 옆에 붙여 이름을 달아 줍시다.

**요구사항**:
- `data/titanic.csv` 는 변수 `df` 에, `data/port_info.csv` 는 변수 `port` 에 각각 담으세요.
- `embarked` 를 기준으로 `df` 에 `port` 를 `how='left'` 로 병합해 `merged` 에 담으세요.
- 왼쪽(`df`)의 891행을 모두 유지하고, `port_name` 열이 새로 생깁니다.

**예시**
```
merged.shape[0]                          →  891
'port_name' in merged.columns            →  True
(merged['port_name'] == 'Southampton').sum()  →  644
```
<details><summary>힌트</summary>

```text
접근방법:
- 공통 열(embarked)을 열쇠로 두 표를 옆으로 잇는다. how='left' 는 왼쪽 표를 다 남긴다.

세부구현:
1. 두 파일을 각각 df, port 로 불러온다
2. merge 로 두 표를 embarked 열을 열쇠로 잇되, how 는 left 로 왼쪽 표를 다 남긴다
3. 결과를 merged 에 담는다 (왼쪽 891행 유지, port_name 열 생김)
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day07_EDA_시각화/data/titanic.csv')
port = pd.read_csv('../../day07_EDA_시각화/data/port_info.csv')
merged = df.merge(port, on='embarked', how='left')
print(merged.shape)
print(merged[['embarked', 'port_name']].head())

In [ ]:
# [자가채점]
assert merged.shape[0] == 891
assert 'port_name' in merged.columns
assert (merged['port_name'] == 'Southampton').sum() == 644
print("✅ 문제6 통과!")

### 해설 — 문제 6
- **접근법**: `merge` 는 공통 열(`on='embarked'`)을 열쇠로 두 표를 **옆으로**(열 방향) 잇습니다. `how='left'` 는 왼쪽 표의 행을 모두 유지해요.
- **흔한 실수**: `how` 를 기본값(`inner`)으로 두면 항구 코드가 빈(결측) 2행이 사라져 889행이 됩니다. 왼쪽을 다 남기려면 `how='left'`.
- **대안**: 두 표의 열쇠 이름이 다르면 `left_on`·`right_on` 으로 각각 지정합니다.

## 7. 표 붙이기 — 행 방향 (concat)
**배경**: 1등석 승객표와 2등석 승객표를 따로 만들었다면, 둘을 **위아래로 이어 붙여** 하나로 합칠 수 있습니다.

**요구사항**:
- `data/titanic.csv` 파일을 읽어 변수 `df` 에 담으세요.
- `pclass == 1` 인 행만 골라 `first`, `pclass == 2` 인 행만 골라 `second` 에 담으세요.
- `pd.concat` 으로 `first` 와 `second` 를 **세로로** 이어 붙여 `combined` 에 담으세요. (`ignore_index=True`)

**예시**
```
len(combined)  →  400   (1등석 216 + 2등석 184)
```
<details><summary>힌트</summary>

```text
접근방법:
- 두 표를 리스트로 묶어 concat 하면 위아래로 이어 붙는다.

세부구현:
1. 파일을 df 로 불러온다
2. 불리언 필터로 first(1등석), second(2등석) 를 만든다
3. concat 으로 first·second 두 조각을 위아래로 이어 붙이고(인덱스는 새로 매김) combined 에 담는다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day07_EDA_시각화/data/titanic.csv')
first = df[df['pclass'] == 1]
second = df[df['pclass'] == 2]
combined = pd.concat([first, second], ignore_index=True)
print(len(combined))

In [ ]:
# [자가채점]
assert len(combined) == 400
assert set(combined['pclass'].unique()) == {1, 2}
print("✅ 문제7 통과!")

### 해설 — 문제 7
- **접근법**: `pd.concat([a, b])` 은 여러 표를 <strong>위아래(행 방향)</strong>로 이어 붙입니다. `ignore_index=True` 는 인덱스를 0부터 새로 매겨 겹침을 막아요.
- **흔한 실수**: 표를 리스트로 감싸지 않고 `pd.concat(first, second)` 로 부르면 에러가 납니다. 반드시 `[first, second]` 처럼 목록으로 주세요.
- **대안**: 옆으로(열 방향) 붙이려면 `axis=1` 을 주지만, 같은 열을 가진 표를 쌓을 땐 기본값(`axis=0`)이 맞습니다.

## 8. 범주 개수 막대 (countplot)
**배경**: 각 객실 등급에 몇 명이 탔는지 막대로 세어 보면 3등석이 압도적으로 많음을 한눈에 볼 수 있습니다.

**요구사항**:
- `data/titanic.csv` 파일을 읽어 변수 `df` 에 담으세요.
- `sns.countplot` 으로 `x='pclass'` 인 막대그래프를 그리고, 결과 Axes 를 `ax` 에 저장하세요.
- `ax.set_title('등급별 인원')` 처럼 **제목을 다세요**. 끝에 `plt.show()` 를 호출하세요.

**예시**: x축이 `pclass`, 각 등급의 막대 높이가 인원 수. 제목이 비어 있지 않아야 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- countplot 은 범주별 행 개수를 세어 막대로 그린다. 반환된 Axes 를 ax 에 담아 제목을 단다.

세부구현:
1. 파일을 df 로 불러온다
2. countplot 으로 등급(pclass)별 개수를 그리고 결과 Axes 를 ax 에 담는다
3. ax 에 set_title 으로 제목을 달고 plt.show 로 그래프를 보여 준다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day07_EDA_시각화/images/과제/lv1_q8.png" width="520"/>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../../day07_EDA_시각화/data/titanic.csv')
plt.figure()
ax = sns.countplot(data=df, x='pclass')
ax.set_title('등급별 인원')
plt.show()

### 해설 — 문제 8
- **접근법**: `countplot` 은 지정한 범주 열의 **행 개수**를 세어 막대로 그립니다. 미리 집계할 필요 없이 원본을 그대로 넘겨요.
- **흔한 실수**: seaborn 축-레벨 함수는 Axes 를 돌려줍니다. 이를 `ax` 로 받지 않으면 `set_title` 을 걸 곳이 없어요.
- **대안**: y축에 범주를 두어 가로 막대로 그리려면 `y='pclass'` 로 바꿉니다.

## 9. 그룹 평균 막대 (barplot)
**배경**: 탑승 항구별로 평균 요금이 얼마나 달랐는지 막대로 비교해 봅시다. `barplot` 은 그룹별 평균을 자동으로 계산해 그립니다.

**요구사항**:
- `data/titanic.csv` 파일을 읽어 변수 `df` 에 담으세요.
- `sns.barplot` 으로 `x='embarked'`, `y='fare'` 인 막대그래프를 그리세요. 이때 **`errorbar=None`** 을 주세요.
- 결과 Axes 를 `ax` 에 저장하고 `ax.set_title(...)` 로 제목을 다세요. 끝에 `plt.show()`.

**예시**: x축 `embarked`, y축 `fare`, 항구별 평균 요금 막대. 제목이 비어 있지 않아야 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- barplot 은 x 범주별 y 평균을 막대로 그린다. errorbar=None 으로 오차막대를 끈다.

세부구현:
1. 파일을 df 로 불러온다
2. barplot 으로 x 는 항구(embarked), y 는 요금(fare)을 그리되 오차막대는 끈다(errorbar=None). 결과 Axes 를 ax 에 담는다
3. ax 에 set_title 으로 제목을 달고 plt.show 로 그래프를 보여 준다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day07_EDA_시각화/images/과제/lv1_q9.png" width="520"/>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../../day07_EDA_시각화/data/titanic.csv')
plt.figure()
ax = sns.barplot(data=df, x='embarked', y='fare', errorbar=None)
ax.set_title('항구별 평균 요금')
plt.show()

### 해설 — 문제 9
- **접근법**: `barplot` 은 `x` 범주별로 `y` 의 **평균**을 자동 계산해 막대로 그립니다. 미리 groupby 하지 않아도 돼요.
- **흔한 실수**: `errorbar=None` 을 빼면 막대 위에 얇은 세로선(오차막대)이 함께 그려집니다 — 이 단원에서는 평균 막대만 보면 되니 꼭 꺼 주세요.
- **대안**: 평균이 아닌 합계 막대를 원하면 `estimator='sum'` 을 줍니다.

## 10. 분포 — 히스토그램 (histplot)
**배경**: 승객 나이가 어떻게 퍼져 있는지(젊은 층이 많은지) 보려면 **분포**를 봅니다. 히스토그램은 값의 구간별 개수를 막대로 나타냅니다.

**요구사항**:
- `data/titanic.csv` 파일을 읽어 변수 `df` 에 담으세요.
- `sns.histplot` 으로 `x='age'`, `bins=20` 인 히스토그램을 그리세요.
- 결과 Axes 를 `ax` 에 저장하고 `ax.set_title(...)` 로 제목을 다세요. 끝에 `plt.show()`.

**예시**: x축이 `age`, 20개 구간의 막대. 제목이 비어 있지 않아야 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- histplot 은 한 숫자 열을 구간(bins)으로 나눠 개수를 막대로 그린다.

세부구현:
1. 파일을 df 로 불러온다
2. histplot 으로 나이(age)를 20개 구간(bins)으로 그리고 결과 Axes 를 ax 에 담는다
3. ax 에 set_title 으로 제목을 달고 plt.show 로 그래프를 보여 준다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day07_EDA_시각화/images/과제/lv1_q10.png" width="520"/>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../../day07_EDA_시각화/data/titanic.csv')
plt.figure()
ax = sns.histplot(data=df, x='age', bins=20)
ax.set_title('나이 분포')
plt.show()

### 해설 — 문제 10
- **접근법**: `histplot` 은 한 숫자 열을 여러 구간(`bins`)으로 나눠 각 구간의 개수를 막대로 그립니다. 분포의 모양(치우침·봉우리)을 봅니다.
- **흔한 실수**: `bins` 수를 너무 크게/작게 잡으면 모양이 왜곡됩니다. 여기선 문제 지시대로 20으로 두세요.
- **대안**: 매끄러운 곡선으로 분포를 보고 싶으면 다음 문제의 `kdeplot` 을 씁니다.

## 11. 분포 — 상자그림 (boxplot)
**배경**: 등급별 요금 분포를 중앙값·사분위수·이상치까지 한 상자로 요약해 비교하려면 상자그림(boxplot)이 좋습니다.

**요구사항**:
- `data/titanic.csv` 파일을 읽어 변수 `df` 에 담으세요.
- `sns.boxplot` 으로 `x='pclass'`, `y='fare'` 인 상자그림을 그리세요.
- 결과 Axes 를 `ax` 에 저장하고 `ax.set_title(...)` 로 제목을 다세요. 끝에 `plt.show()`.

**예시**: x축 `pclass`, y축 `fare`, 등급마다 상자 하나. 제목이 비어 있지 않아야 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- boxplot 은 x 범주별로 y 값의 중앙값·사분위·이상치를 상자로 요약해 그린다.

세부구현:
1. 파일을 df 로 불러온다
2. boxplot 으로 x 는 등급(pclass), y 는 요금(fare)을 그리고 결과 Axes 를 ax 에 담는다
3. ax 에 set_title 으로 제목을 달고 plt.show 로 그래프를 보여 준다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day07_EDA_시각화/images/과제/lv1_q11.png" width="520"/>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../../day07_EDA_시각화/data/titanic.csv')
plt.figure()
ax = sns.boxplot(data=df, x='pclass', y='fare')
ax.set_title('등급별 요금 분포')
plt.show()

### 해설 — 문제 11
- **접근법**: 상자그림은 상자(사분위 범위)·가운데 선(중앙값)·수염(범위)·점(이상치)으로 분포를 요약합니다. 여러 그룹을 나란히 비교하기 좋아요.
- **흔한 실수**: `x` 와 `y` 를 바꾸면 상자가 가로로 눕습니다. 등급을 가로축에 두려면 `x='pclass'`, `y='fare'` 순서를 지키세요.
- **대안**: 분포 모양까지 더 자세히 보고 싶으면 `violinplot` 을 씁니다.

## 12. 분포 — 밀도 곡선 (kdeplot)
**배경**: 히스토그램의 막대를 매끄러운 곡선으로 바꾼 것이 밀도 곡선(KDE)입니다. 생존자와 사망자의 나이 분포를 겹쳐 비교해 봅시다.

**요구사항**:
- `data/titanic.csv` 파일을 읽어 변수 `df` 에 담으세요.
- `sns.kdeplot` 으로 `x='age'`, `hue='survived'` 인 밀도 곡선을 그리세요. (`hue` 로 생존여부별 곡선이 나뉩니다)
- 결과 Axes 를 `ax` 에 저장하고 `ax.set_title(...)` 로 제목을 다세요. 끝에 `plt.show()`.

**예시**: x축이 `age`, 생존여부(0·1)별 두 밀도 곡선. 제목이 비어 있지 않아야 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- kdeplot 은 분포를 매끄러운 곡선으로 그린다. hue 를 주면 그룹마다 곡선이 나뉜다.

세부구현:
1. 파일을 df 로 불러온다
2. kdeplot 으로 나이(age)를 그리되 hue 에 survived 를 주어 생존여부별 곡선으로 나눈다. 결과 Axes 를 ax 에 담는다
3. ax 에 set_title 으로 제목을 달고 plt.show 로 그래프를 보여 준다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day07_EDA_시각화/images/과제/lv1_q12.png" width="520"/>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../../day07_EDA_시각화/data/titanic.csv')
plt.figure()
ax = sns.kdeplot(data=df, x='age', hue='survived')
ax.set_title('생존여부별 나이 분포')
plt.show()

### 해설 — 문제 12
- **접근법**: `kdeplot` 은 분포를 매끄러운 곡선(밀도)으로 그립니다. `hue='survived'` 를 주면 생존/사망 곡선이 색으로 나뉘어 겹쳐 보여요.
- **흔한 실수**: `hue` 에 숫자 열을 주면 seaborn 이 범주로 취급합니다(0·1 두 곡선). 연속값을 `hue` 로 주면 곡선이 너무 많아지니 주의하세요.
- **대안**: 곡선과 막대를 함께 보려면 `histplot(..., kde=True)` 로 히스토그램 위에 곡선을 얹을 수 있습니다.

## 13. 두 수치의 관계 (scatterplot)
**배경**: 나이(`age`)와 요금(`fare`)이 어떤 관계를 보이는지, 생존여부까지 색으로 나눠 점으로 흩뿌려 봅니다.

**요구사항**:
- `data/titanic.csv` 파일을 읽어 변수 `df` 에 담으세요.
- `sns.scatterplot` 으로 `x='age'`, `y='fare'`, `hue='survived'` 인 산점도를 그리세요.
- 결과 Axes 를 `ax` 에 저장하고 `ax.set_title(...)` 로 제목을 다세요. 끝에 `plt.show()`.

**예시**: x축 `age`, y축 `fare`, 점 색이 생존여부. 제목이 비어 있지 않아야 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- scatterplot 은 두 숫자 열을 x·y 좌표로 점을 찍는다. hue 로 점 색을 그룹별로 나눈다.

세부구현:
1. 파일을 df 로 불러온다
2. scatterplot 으로 x 는 나이(age), y 는 요금(fare)을 찍되 hue 에 survived 를 주어 색을 나눈다. 결과 Axes 를 ax 에 담는다
3. ax 에 set_title 으로 제목을 달고 plt.show 로 그래프를 보여 준다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day07_EDA_시각화/images/과제/lv1_q13.png" width="520"/>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../../day07_EDA_시각화/data/titanic.csv')
plt.figure()
ax = sns.scatterplot(data=df, x='age', y='fare', hue='survived')
ax.set_title('나이와 요금')
plt.show()

### 해설 — 문제 13
- **접근법**: 산점도는 두 숫자 열을 x·y 좌표로 점을 찍어 관계를 눈으로 봅니다. `hue` 로 세 번째 범주(생존여부)를 색으로 얹었어요.
- **흔한 실수**: 점이 너무 겹쳐 보기 어려우면 `alpha=0.5`(투명도)를 줘 겹침을 완화할 수 있습니다.
- **대안**: 여러 숫자 열의 관계를 한 번에 보려면 `pairplot` 으로 모든 짝을 격자로 그립니다.

## 14. 순서에 따른 흐름 — 선그래프 (lineplot)
**배경**: <strong>나이대(연령대)</strong>가 올라갈수록 평균 요금이 어떻게 달라지는지 **선그래프**로 봅니다. 아래 `# [제공 코드]` 셀이 나이대별 평균 요금표 `fare_by_agegroup` 를 미리 만들어 둡니다.

> 아래 표는 준비해 두었습니다. 여러분은 이 표로 **선그래프만** 그리면 됩니다.

**요구사항**:
- 먼저 아래 `# [제공 코드]` 셀을 실행하면 `fare_by_agegroup` 표(열: `age_group`, `avg_fare`)가 준비됩니다.
- `sns.lineplot` 으로 `data=fare_by_agegroup`, `x='age_group'`, `y='avg_fare'` 인 선그래프를 그리세요.
- 결과 Axes 를 변수 `ax` 에 저장하고 `ax.set_title(...)` 로 제목을 다세요. 끝에 `plt.show()` 를 호출하세요.

**예시**: x축이 나이대(`age_group`), y축이 평균 요금(`avg_fare`)인 선그래프. 제목이 비어 있지 않아야 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 제공된 표를 그대로 선그래프로 그린다. lineplot 의 x 에는 나이대(age_group), y 에는 평균 요금(avg_fare) 열을 준다.

세부구현:
1. 제공된 fare_by_agegroup 표를 lineplot 에 넣는다 (x 는 age_group, y 는 avg_fare)
2. 그린 결과 Axes 를 ax 에 담는다
3. ax 에 set_title 으로 제목을 달고 plt.show 로 그래프를 보여 준다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day07_EDA_시각화/images/과제/lv1_q14.png" width="520"/>

In [ ]:
# [제공 코드] 나이대(연령대)별 평균 요금표를 준비해 둡니다. 여러분은 이 표로 선그래프만 그리면 됩니다.
df = pd.read_csv('../../day07_EDA_시각화/data/titanic.csv')
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80]
labels = ['0-9', '10-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)
fare_by_agegroup = df.groupby('age_group', observed=True)['fare'].mean().reset_index()
fare_by_agegroup.columns = ['age_group', 'avg_fare']
print(fare_by_agegroup)

In [ ]:
plt.figure()
ax = sns.lineplot(data=fare_by_agegroup, x='age_group', y='avg_fare')
ax.set_title('나이대별 평균 요금')
plt.show()

### 해설 — 문제 14
- **접근법**: 나이대별 평균 요금표(`fare_by_agegroup`)는 미리 준비돼 있으니, 이 문제에서는 `lineplot` 으로 **선그래프 그리기 한 가지**에 집중합니다. `x='age_group'`, `y='avg_fare'` 로 넘기면 seaborn 이 축 이름을 자동으로 달아 줍니다.
- **흔한 실수**: seaborn 축-레벨 함수는 Axes 를 돌려줍니다. 이를 `ax` 로 받지 않으면 `set_title` 을 걸 곳이 없어요.
- **대안**: 표를 직접 만드는 과정 — 원본을 그룹으로 묶어 평균을 내고 `reset_index` 로 정리하는 흐름 — 은 LV2 에서 이어서 연습합니다.

## 15. 요약표를 색으로 (heatmap)
**배경**: 문제 4처럼 만든 요약표를 숫자만 보면 크기 비교가 어렵습니다. 히트맵은 값의 크기를 **색 진하기**로 보여 줘 한눈에 비교됩니다.

**요구사항**:
- `data/titanic.csv` 파일을 읽어 변수 `df` 에 담으세요.
- `index='pclass'`, `columns='sex'`, `values='fare'` 로 **평균 요금** 요약표를 만들어 `pv` 에 담으세요.
- `sns.heatmap(pv, annot=True)` 로 히트맵을 그리고, 결과 Axes 를 `ax` 에 저장한 뒤 `ax.set_title(...)` 로 제목을 다세요. 끝에 `plt.show()`.

**예시**
```
pv.shape                     →  (3, 2)
round(pv.loc[1, 'female'], 2) →  106.13   (1등석 여성 평균 요금)
```

<details><summary>힌트</summary>

```text
접근방법:
- pivot_table 로 등급×성별 평균 요금표를 만든 뒤, heatmap 으로 값의 크기를 색으로 그린다. annot=True 는 칸에 숫자를 표시한다.

세부구현:
1. 파일을 df 로 불러온다
2. pivot_table 로 행은 등급(pclass), 열은 성별(sex), 값은 요금(fare) 평균이 되게 요약해 pv 에 담는다
3. heatmap 으로 pv 를 그리되 칸에 숫자를 표시하고(annot=True) 결과 Axes 를 ax 에 담아 set_title 으로 제목을 단다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day07_EDA_시각화/images/과제/lv1_q15.png" width="520"/>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../../day07_EDA_시각화/data/titanic.csv')
pv = df.pivot_table(index='pclass', columns='sex', values='fare')
plt.figure()
ax = sns.heatmap(pv, annot=True)
ax.set_title('등급·성별 평균 요금')
plt.show()

### 해설 — 문제 15
- **접근법**: 먼저 `pivot_table` 로 등급×성별 평균 요금표를 만들고, `heatmap` 으로 값의 크기를 색으로 칠합니다. `annot=True` 는 각 칸에 숫자를 함께 적어 줘요.
- **흔한 실수**: `heatmap` 은 **표 형태(2차원)** 데이터를 받습니다. 원본 `df` 를 그대로 넘기면 안 되고, 반드시 집계표(`pv`)를 넘기세요.
- **대안**: 색 대비를 바꾸려면 `cmap='YlGnBu'` 처럼 색 팔레트를 지정할 수 있습니다. (색상 대비는 취향껏 — 값의 크기 비교가 목적입니다.)

## 16. 그래프에서 관찰한 사실 (서술형)
**배경**: 그래프는 그리는 것보다 **읽는 것**이 중요합니다. 앞의 그래프(문제 8~15) 중 하나를 골라, 그 그림에서 실제로 **관찰한 사실 한 가지**를 문장으로 적어 보세요.

**요구사항**:
- 어떤 그래프(문제 번호)를 근거로 삼았는지 밝히세요.
- 그 그래프에서 눈으로 확인한 사실을 1~2문장으로 서술하세요. (예: "3등석 인원이 가장 많다", "1등석 여성 생존율이 가장 높다" 등)
- 정답은 하나가 아닙니다. 그래프가 실제로 보여 주는 사실이면 됩니다.

**예시**: `*(문제 8 countplot 을 보면 3등석 막대가 가장 높아, 3등석 승객이 가장 많았음을 알 수 있다.)*`

> 이 문제는 자가채점(assert)이 없습니다. 아래 서술 셀에 직접 문장을 적고, 정답 노트북의 모범 서술과 비교해 보세요.

**모범 서술 (예시 — 정답은 여럿)**

- **문제 8(countplot)**: 3등석 막대가 1·2등석보다 훨씬 높아, 3등석 승객(491명)이 가장 많았음을 알 수 있다.
- **문제 11(boxplot)**: 1등석 요금 상자가 2·3등석보다 위쪽에 넓게 퍼져 있어, 등급이 높을수록 요금이 비싸고 편차도 컸음을 볼 수 있다.
- **문제 15(heatmap)**: 1등석 여성 칸(106.13)의 색이 가장 진해, 같은 1등석이라도 여성 승객의 평균 요금이 남성보다 높았음을 알 수 있다.

핵심은 **그래프가 실제로 보여 주는 것**을 근거로 삼는 것입니다. 그림에 없는 값을 지어내지 말고, 눈으로 확인 가능한 사실을 적으면 정답입니다.